In [ ]:
# Bootstrap: make the repo root importable (for `common` and `segmentation`)
# now that this notebook lives in match-context/, one level below repo root.
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

In [ ]:
import pandas as pd
import numpy as np
import os

import floodlight.io.dfl as dfl
from floodlight.models.kinematics import VelocityModel
from floodlight.transforms.filter import butterworth_lowpass

from scipy.optimize import linear_sum_assignment
from scipy.spatial import ConvexHull

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

## Data Loading

Same loading pattern as `segmentation/discretisation.py`.

In [ ]:
from common import config

DATA_DIR = config.DATA_DIR
ls_match_ids = config.DEFAULT_MATCH_IDS

match_id = 'J03WPY'

In [ ]:
path_positions = os.path.join(DATA_DIR, [x for x in os.listdir(DATA_DIR) if match_id in x and 'positions' in x][0])
path_events    = os.path.join(DATA_DIR, [x for x in os.listdir(DATA_DIR) if match_id in x and 'events' in x][0])
path_info      = os.path.join(DATA_DIR, [x for x in os.listdir(DATA_DIR) if match_id in x and 'matchinformation' in x][0])

In [ ]:
xy, possession, ballstatus, teamsheets, pitch = dfl.read_position_data_xml(
    path_positions, path_info, teamsheet_home=None, teamsheet_away=None
)

In [ ]:
events, teamsheets, pitch = dfl.read_event_data_xml(path_events, path_info)

# read_event_data_xml returns fresh Teamsheet objects without xIDs — add them now
for team in ['Home', 'Away']:
    teamsheets[team].add_xIDs()

In [ ]:
# Convenience references — raw numpy arrays
xy_h1_home = xy['firstHalf']['Home'].xy      # shape: (n_frames_h1, 2 * n_home)
xy_h1_away = xy['firstHalf']['Away'].xy
xy_h2_home = xy['secondHalf']['Home'].xy
xy_h2_away = xy['secondHalf']['Away'].xy

# np.array(...) first: floodlight's Code objects (current floodlight version)
# aren't raw ndarrays themselves, unlike xy.xy above.
poss_h1 = np.array(possession['firstHalf']).ravel()    # shape: (n_frames_h1,)
poss_h2 = np.array(possession['secondHalf']).ravel()

bstat_h1 = np.array(ballstatus['firstHalf']).ravel()   # shape: (n_frames_h1,)
bstat_h2 = np.array(ballstatus['secondHalf']).ravel()

n_frames_h1 = len(poss_h1)
n_frames_h2 = len(poss_h2)

print(f"First half:  {n_frames_h1} frames  ({n_frames_h1/25/60:.1f} min)")
print(f"Second half: {n_frames_h2} frames  ({n_frames_h2/25/60:.1f} min)")

## Inspect Possession and Ballstatus Encoding

In [ ]:
print("Possession unique values:", np.unique(poss_h1))
print("Ballstatus unique values:", np.unique(bstat_h1))

# Expected:
#   possession:  1 = Home, 2 = Away  (0 or other = contested / no clear possession)
#   ballstatus:  1 = ball in play,   0 = ball out of play

In [ ]:
# Map possession integers to team labels
# Adjust POSS_MAP if your match shows different values above
POSS_MAP   = {1: 'Home', 2: 'Away'}   # 0 / other = 'contest'
BSTAT_LIVE = 1                         # ballstatus value that means ball is active

def decode_possession(val):
    return POSS_MAP.get(int(val), 'contest')

## Build Full-Match Arrays

Concatenate both halves into a single array with a global frame index.  
All subsequent per-frame computations operate on these combined arrays.

**DFL coordinate system** (unchanged between halves in the raw data):  
x ∈ [-52, +52] (left → right), y ∈ [-34, +34] (bottom → top)  
Attack direction is encoded separately in `ATTACK_DIRECTION`.

In [ ]:
n_home = xy_h1_home.shape[1] // 2
n_away = xy_h1_away.shape[1] // 2
n_total = n_frames_h1 + n_frames_h2

# Full-match arrays
xy_home_full = np.concatenate([xy_h1_home, xy_h2_home], axis=0)   # (n_total, 2*n_home)
xy_away_full = np.concatenate([xy_h1_away, xy_h2_away], axis=0)
poss_full    = np.concatenate([poss_h1, poss_h2], axis=0)          # (n_total,)
bstat_full   = np.concatenate([bstat_h1, bstat_h2], axis=0)

# Half label per frame (used to look up attack direction)
half_labels = np.array(['firstHalf'] * n_frames_h1 + ['secondHalf'] * n_frames_h2)

print(f"Full-match arrays: {n_total} frames, {n_home} Home players, {n_away} Away players")

## Possession Phase Segmentation

Following Llana et al. (2022) §2.2: a **possession phase** is a maximal contiguous sequence of frames  
with the same possession team *and* the same ball-in-play status. A new phase begins whenever  
possession changes, the ball goes dead, or the ball returns to play.

In [ ]:
def segment_possession_phases(poss_arr, bstat_arr, half_labels, framerate=25):
    """
    Segment the full match into discrete possession phases.

    A phase changes when possession or ballstatus changes between consecutive frames.

    Returns
    -------
    pd.DataFrame
        One row per phase with columns:
        phase_id, start_frame, end_frame, half, possession_team,
        ball_live, duration_frames, duration_s
    """
    n = len(poss_arr)

    # Detect change points: frames where either signal differs from the previous
    change = np.zeros(n, dtype=bool)
    change[0] = True
    change[1:] = (poss_arr[1:] != poss_arr[:-1]) | (bstat_arr[1:] != bstat_arr[:-1])

    starts = np.where(change)[0]
    ends   = np.concatenate([starts[1:] - 1, [n - 1]])

    rows = []
    for pid, (s, e) in enumerate(zip(starts, ends)):
        rows.append({
            'phase_id':        pid,
            'start_frame':     int(s),
            'end_frame':       int(e),
            'half':            half_labels[s],
            'possession_team': decode_possession(poss_arr[s]),
            'ball_live':       bool(bstat_arr[s] == BSTAT_LIVE),
            'duration_frames': int(e - s + 1),
            'duration_s':      (e - s + 1) / framerate,
        })

    return pd.DataFrame(rows)


df_phases = segment_possession_phases(poss_full, bstat_full, half_labels)
print(f"Total phases: {len(df_phases)}")
df_phases.head(10)

In [ ]:
print(df_phases.groupby(['possession_team', 'ball_live'])['duration_s'].describe().round(2))

## Attack Direction

In the DFL data, the raw coordinates are **not** flipped between halves — the coordinate
system is fixed to the stadium, and which side attacks which direction is **match-specific**
(it depends on the stadium/broadcast setup, not a fixed convention). The attack direction is
therefore looked up per match from the same `direction_idsse_videos.json` file used by
`segmentation/discretisation.py` (via its `get_play_direction()`), rather than assumed fixed
across all matches.

`get_attack_sign(half, team)` returns +1 if the team attacks toward increasing x, −1 otherwise.

In [ ]:
import json as _json
from common import config as _config
from segmentation.discretisation import get_play_direction

with open(_config.DIRECTION_JSON, "r") as _f:
    dict_direction = _json.load(_f)

# Fallback convention (only used if match_id/half is missing from the JSON),
# matching the ATTACK_SIGN convention this notebook used before per-match
# lookup was added -- kept only so this notebook still runs on matches
# without direction data, with an explicit warning rather than a silent guess.
_FALLBACK_ATTACK_SIGN = {
    'firstHalf':  {'Home': +1, 'Away': -1},
    'secondHalf': {'Home': -1, 'Away': +1},
}


def get_attack_sign(half, team):
    """
    +1 if `team` attacks toward increasing x during `half`, −1 otherwise.
    Looked up per match from dict_direction (see segmentation.discretisation.get_play_direction);
    falls back to a fixed convention with a warning if the match/half is unknown.
    """
    direction = get_play_direction(dict_direction, match_id, team, half)
    if direction is None:
        print(f"[WARN] No direction data for match={match_id}, team={team}, half={half} "
              "-- using fallback convention.")
        return _FALLBACK_ATTACK_SIGN[half][team]
    return 1 if direction == 'left_to_right' else -1


def normalize_x_to_attack(x_arr, half, team):
    """
    Return x-coordinates in the team's attacking coordinate system:
    positive = closer to opponent's goal, negative = closer to own goal.
    """
    return x_arr * get_attack_sign(half, team)

## Per-Frame Block Features

For each frame we compute:
1. **Block centroid** — mean (x, y) of all outfield players of each team.
2. **Defensive lines** — the defending team's outfield players are sorted by x and split
   into three equally-sized groups (back line / mid line / press line). The mean x of each
   group gives the line position (Llana et al. 2022, §2.3).
3. **Defense type** — based on the defending block's centroid relative to the halfway line,
   expressed in the attacking team's coordinate system.

In [ ]:
def get_gk_xids(teamsheet_df):
    """Return xID values of players with goalkeeper position ('TW')."""
    return set(teamsheet_df.loc[teamsheet_df['position'] == 'TW', 'xID'].values)


def get_outfield_xids(teamsheet_df):
    """Return xID values of all outfield players (non-GK)."""
    return set(teamsheet_df.loc[teamsheet_df['position'] != 'TW', 'xID'].values)


def extract_player_positions(xy_frame, xids):
    """
    Extract (x, y) positions for the given xIDs from a single tracking frame.

    Parameters
    ----------
    xy_frame : np.ndarray  shape (2 * n_players,)
        One row of the XY array: [x0, y0, x1, y1, ...]
    xids : iterable of int
        0-based xIDs (column indices) for the desired players.

    Returns
    -------
    np.ndarray  shape (n_valid, 2)  — NaN rows dropped.
    """
    rows = []
    for xid in xids:
        x = xy_frame[2 * xid]
        y = xy_frame[2 * xid + 1]
        if not (np.isnan(x) or np.isnan(y)):
            rows.append([x, y])
    return np.array(rows) if rows else np.empty((0, 2))


def block_centroid(positions):
    """Mean (x, y) of a set of player positions; returns (NaN, NaN) if empty."""
    if len(positions) == 0:
        return np.nan, np.nan
    return float(np.nanmean(positions[:, 0])), float(np.nanmean(positions[:, 1]))


def defensive_lines(positions, attack_sign, n_lines=3):
    """
    Compute x-positions of n_lines defensive lines for the defending team.

    Outfield players are sorted by x in the *defending* direction (deepest to
    highest) and split into n_lines equal groups. Returns line centroids in
    raw (pitch) x-coordinates, from deepest to shallowest.

    attack_sign : +1 or −1  — the *attacking* team's sign; defender's sign is opposite.
    """
    if len(positions) < n_lines:
        return [np.nan] * n_lines

    # Sort x from defensive-deepest to defensive-shallowest
    # Defender's own goal is in the direction of -attack_sign
    # "Deepest" defender = farthest from attacking goal = smallest attack_sign*x value
    x = positions[:, 0]
    sorted_x = np.sort(x)[::int(-attack_sign)]  # reverse sort if attack_sign=+1 → deepest last
    # Actually: deepest defender has the smallest x*attack_sign (most behind)
    # Sort ascending in attack-coordinate then groups go: [back, mid, press]
    attack_x = x * attack_sign
    sorted_idx = np.argsort(attack_x)       # ascending attack_x: back line first
    sorted_raw_x = x[sorted_idx]

    groups = np.array_split(sorted_raw_x, n_lines)
    return [float(g.mean()) for g in groups]


def classify_defense_type(defending_centroid_x, attack_sign_of_opponent):
    """
    Classify the defending team's block organization.

    Parameters
    ----------
    defending_centroid_x : float
        Raw x-coordinate of the defending team's block centroid.
    attack_sign_of_opponent : +1 or −1
        +1 if the attacking (opposing) team attacks toward positive x.

    Returns
    -------
    'high_press' | 'medium_block' | 'low_block'

    Logic (Llana 2022 §2.3):
    - high_press:   defending block is in the attacking team's half
                    (centroid on the same side as the attacking goal)
    - low_block:    defending block is well inside the defending team's own half
    - medium_block: everything in between
    """
    if np.isnan(defending_centroid_x):
        return np.nan
    # Project centroid onto attacking axis; threshold at 0 (halfway) and ±15 m
    atk_x = defending_centroid_x * attack_sign_of_opponent
    if atk_x > 0:
        return 'high_press'
    elif atk_x > -15:
        return 'medium_block'
    else:
        return 'low_block'

In [ ]:
# Pre-compute xID sets for each team
home_outfield_xids = get_outfield_xids(teamsheets['Home'].teamsheet)
away_outfield_xids = get_outfield_xids(teamsheets['Away'].teamsheet)
home_all_xids      = set(teamsheets['Home'].teamsheet['xID'].values)
away_all_xids      = set(teamsheets['Away'].teamsheet['xID'].values)

print(f"Home outfield xIDs: {sorted(home_outfield_xids)}")
print(f"Away outfield xIDs: {sorted(away_outfield_xids)}")

In [ ]:
# -----------------------------------------------------------------
# Compute per-frame block features for every frame in the match
# -----------------------------------------------------------------
# Pre-allocate result columns
block_cx_home   = np.full(n_total, np.nan)
block_cy_home   = np.full(n_total, np.nan)
block_cx_away   = np.full(n_total, np.nan)
block_cy_away   = np.full(n_total, np.nan)

def_line_home   = np.full((n_total, 3), np.nan)   # 3 line x-positions for Home
def_line_away   = np.full((n_total, 3), np.nan)

defense_type_arr = np.full(n_total, None, dtype=object)

for f in range(n_total):
    half = half_labels[f]
    pteam = decode_possession(poss_full[f])

    # Extract outfield positions for both teams
    home_pos = extract_player_positions(xy_home_full[f], home_outfield_xids)
    away_pos = extract_player_positions(xy_away_full[f], away_outfield_xids)

    # Block centroids
    block_cx_home[f], block_cy_home[f] = block_centroid(home_pos)
    block_cx_away[f], block_cy_away[f] = block_centroid(away_pos)

    # Defensive lines & defense type depend on which team is defending
    if pteam == 'Home':
        # Away is defending; Home is attacking
        atk_sign = get_attack_sign(half, 'Home')
        def_line_away[f] = defensive_lines(away_pos, atk_sign)
        defense_type_arr[f] = classify_defense_type(block_cx_away[f], atk_sign)
    elif pteam == 'Away':
        atk_sign = get_attack_sign(half, 'Away')
        def_line_home[f] = defensive_lines(home_pos, atk_sign)
        defense_type_arr[f] = classify_defense_type(block_cx_home[f], atk_sign)
    # else: contested possession — skip

print("Per-frame block features computed.")

## Phase-Level Attack Type Classification

Each live-ball possession phase is classified by attack type following Llana et al. (2022) §2.2:

| Type | Condition |
|------|-----------|
| `set_piece` | Previous phase was dead-ball *and* ball came alive in the opponent's half |
| `counter_attack` | Previous phase belonged to the *other* team *and* lasted < 5 s |
| `organized_attack` | Default for all other live phases |
| `dead_ball` | Ball-out-of-play phases (no attack type) |

In [ ]:
COUNTER_THRESHOLD_S = 5.0   # phases shorter than this can trigger a counter


def classify_attack_types(df_phases, xy_home_full, xy_away_full, half_labels):
    """
    Classify each possession phase by attack type.

    A 'set_piece' is a live phase that immediately follows a dead-ball phase.
    A 'counter_attack' is a live phase that immediately follows a short (< threshold)
    live phase by the opposing team.
    """
    attack_types = []

    for i, row in df_phases.iterrows():
        if not row['ball_live']:
            attack_types.append('dead_ball')
            continue

        if row['possession_team'] == 'contest':
            attack_types.append('contest')
            continue

        # Look at the previous phase
        prev = df_phases.iloc[i - 1] if i > 0 else None

        if prev is not None and not prev['ball_live']:
            # Immediately follows a dead ball → set piece
            attack_types.append('set_piece')

        elif (prev is not None
              and prev['ball_live']
              and prev['possession_team'] != row['possession_team']
              and prev['possession_team'] not in ('contest', None)
              and prev['duration_s'] < COUNTER_THRESHOLD_S):
            # Ball recovered from a short opposing possession → counter attack
            attack_types.append('counter_attack')

        else:
            attack_types.append('organized_attack')

    return attack_types


df_phases['attack_type'] = classify_attack_types(
    df_phases, xy_home_full, xy_away_full, half_labels
)

print(df_phases['attack_type'].value_counts())

## Player Role Assignment per Possession Phase

Following Llana et al. (2022) §2.4, player roles are assigned dynamically:  
1. Compute each player's **mean position** during the possession phase.  
2. **Normalize** positions to the team's attack direction (so x increases toward the opponent goal).  
3. Run the **Hungarian algorithm** (`linear_sum_assignment`) to optimally match players to  
   the nearest positions in the best-fitting formation template.  
4. Map template slots to role labels (GK, CB, FB, DM, CM, Winger, Striker).

Formation templates are defined in normalized pitch coordinates:  
- x ∈ [0, 1]: 0 = own goal line, 1 = opponent goal line  
- y ∈ [0, 1]: 0 = bottom touchline, 1 = top touchline

In [ ]:
# Formation templates: list of (norm_x, norm_y, role_label) for 11 players
# Sorted deepest → highest (GK first)

FORMATIONS = {
    '4-4-2': [
        (0.04, 0.50, 'GK'),
        (0.28, 0.20, 'FB'), (0.28, 0.40, 'CB'), (0.28, 0.60, 'CB'), (0.28, 0.80, 'FB'),
        (0.55, 0.20, 'Winger'), (0.55, 0.40, 'CM'), (0.55, 0.60, 'CM'), (0.55, 0.80, 'Winger'),
        (0.80, 0.40, 'Striker'), (0.80, 0.60, 'Striker'),
    ],
    '4-3-3': [
        (0.04, 0.50, 'GK'),
        (0.28, 0.20, 'FB'), (0.28, 0.40, 'CB'), (0.28, 0.60, 'CB'), (0.28, 0.80, 'FB'),
        (0.50, 0.25, 'CM'), (0.50, 0.50, 'DM'), (0.50, 0.75, 'CM'),
        (0.80, 0.20, 'Winger'), (0.80, 0.50, 'Striker'), (0.80, 0.80, 'Winger'),
    ],
    '4-2-3-1': [
        (0.04, 0.50, 'GK'),
        (0.28, 0.20, 'FB'), (0.28, 0.40, 'CB'), (0.28, 0.60, 'CB'), (0.28, 0.80, 'FB'),
        (0.45, 0.35, 'DM'), (0.45, 0.65, 'DM'),
        (0.65, 0.20, 'Winger'), (0.65, 0.50, 'CM'), (0.65, 0.80, 'Winger'),
        (0.82, 0.50, 'Striker'),
    ],
    '3-5-2': [
        (0.04, 0.50, 'GK'),
        (0.28, 0.25, 'CB'), (0.28, 0.50, 'CB'), (0.28, 0.75, 'CB'),
        (0.50, 0.10, 'FB'), (0.50, 0.35, 'CM'), (0.50, 0.50, 'DM'), (0.50, 0.65, 'CM'), (0.50, 0.90, 'FB'),
        (0.80, 0.35, 'Striker'), (0.80, 0.65, 'Striker'),
    ],
}

print("Formation templates defined:", list(FORMATIONS.keys()))

In [ ]:
def normalize_positions_to_attack(positions_xy, attack_sign, pitch_length=104, pitch_width=68):
    """
    Normalize raw (x, y) positions to [0, 1] x [0, 1] in the team's attack direction.

    Parameters
    ----------
    positions_xy : np.ndarray  (n, 2)  raw pitch coordinates
    attack_sign  : +1 if team attacks toward +x, −1 toward −x

    Returns
    -------
    np.ndarray  (n, 2)  in [0, 1]
    """
    x_raw, y_raw = positions_xy[:, 0], positions_xy[:, 1]

    # Map x from pitch coords to [0,1] in attacking direction
    half_len = pitch_length / 2   # = 52
    if attack_sign == +1:
        norm_x = (x_raw + half_len) / pitch_length
    else:
        norm_x = (half_len - x_raw) / pitch_length

    norm_y = (y_raw + pitch_width / 2) / pitch_width  # [0, 1], bottom to top

    return np.column_stack([norm_x, norm_y])


def assign_roles_hungarian(mean_pos_norm, formation_template):
    """
    Match n players to the n closest slots in a formation template.

    Parameters
    ----------
    mean_pos_norm  : np.ndarray  (n, 2)  normalized mean positions
    formation_template : list of (norm_x, norm_y, role_label)

    Returns
    -------
    list of str  — role for each player in the order they appear in mean_pos_norm
    """
    n_players = len(mean_pos_norm)
    template_coords = np.array([[t[0], t[1]] for t in formation_template])
    template_roles  = [t[2] for t in formation_template]

    # Use only the first n_players slots if we have fewer players than template
    k = min(n_players, len(template_coords))

    # Cost matrix: Euclidean distance from each player to each template slot
    diffs = mean_pos_norm[:k, np.newaxis, :] - template_coords[np.newaxis, :k, :]
    cost  = np.linalg.norm(diffs, axis=2)   # shape (k, k)

    row_ind, col_ind = linear_sum_assignment(cost)

    roles = ['unknown'] * n_players
    for r, c in zip(row_ind, col_ind):
        roles[r] = template_roles[c]

    return roles


def best_formation_and_roles(mean_pos_norm, formations):
    """
    Try all formation templates and return the one with minimum total assignment cost.
    """
    best_cost   = np.inf
    best_roles  = None
    best_form   = None
    n_players   = len(mean_pos_norm)

    for fname, template in formations.items():
        k = min(n_players, len(template))
        tcoords = np.array([[t[0], t[1]] for t in template[:k]])
        diffs   = mean_pos_norm[:k, np.newaxis, :] - tcoords[np.newaxis, :, :]
        cost    = np.linalg.norm(diffs, axis=2)
        row_ind, col_ind = linear_sum_assignment(cost)
        total_cost = cost[row_ind, col_ind].sum()

        if total_cost < best_cost:
            best_cost  = total_cost
            best_form  = fname
            best_roles = assign_roles_hungarian(mean_pos_norm, template)

    return best_form, best_roles

In [ ]:
def compute_player_roles_for_phases(df_phases, xy_home_full, xy_away_full,
                                     half_labels, teamsheets, formations,
                                     min_phase_s=2.0, framerate=25):
    """
    Compute player roles for every live possession phase using the Hungarian algorithm.

    For each phase, the mean (x, y) position of each player with ≥50% coverage is
    computed, normalized to the team's attack direction, then matched to the best
    formation template.

    Returns
    -------
    pd.DataFrame  columns: phase_id, team, xID, player, role, formation, mean_x, mean_y
    """
    rows = []

    for _, phase in df_phases.iterrows():
        if not phase['ball_live']:
            continue
        if phase['possession_team'] not in ('Home', 'Away'):
            continue
        if phase['duration_s'] < min_phase_s:
            continue

        team  = phase['possession_team']
        half  = phase['half']
        s, e  = int(phase['start_frame']), int(phase['end_frame']) + 1
        atk_sign = get_attack_sign(half, team)
        ts = teamsheets[team].teamsheet

        xy_team = xy_home_full if team == 'Home' else xy_away_full
        phase_xy = xy_team[s:e]   # (n_frames_in_phase, 2*n_players)

        # Mean position per player (skip players with < 50% data coverage)
        mean_positions = []
        valid_xids     = []

        for _, player_row in ts.iterrows():
            xid = int(player_row['xID'])
            px  = phase_xy[:, 2 * xid]
            py  = phase_xy[:, 2 * xid + 1]
            coverage = np.mean(~np.isnan(px))
            if coverage >= 0.5:
                mean_positions.append([np.nanmean(px), np.nanmean(py)])
                valid_xids.append(xid)

        if len(mean_positions) < 4:
            continue

        mean_pos_arr  = np.array(mean_positions)
        mean_pos_norm = normalize_positions_to_attack(mean_pos_arr, atk_sign)

        formation_name, role_list = best_formation_and_roles(mean_pos_norm, formations)

        for xid, (mx, my), norm_pos, role in zip(
                valid_xids, mean_pos_arr, mean_pos_norm, role_list):
            pinfo = ts.loc[ts['xID'] == xid]
            rows.append({
                'phase_id':   phase['phase_id'],
                'team':       team,
                'xID':        xid,
                'player':     pinfo['player'].values[0] if not pinfo.empty else np.nan,
                'role':       role,
                'formation':  formation_name,
                'mean_x':     mx,
                'mean_y':     my,
                'mean_norm_x': norm_pos[0],
                'mean_norm_y': norm_pos[1],
            })

    return pd.DataFrame(rows)


print("Computing player roles per phase (this may take ~1 min)...")
df_player_roles = compute_player_roles_for_phases(
    df_phases, xy_home_full, xy_away_full,
    half_labels, teamsheets, FORMATIONS
)
print(f"Player-role rows: {len(df_player_roles)}")
df_player_roles.groupby(['team', 'role']).size().unstack(fill_value=0)

## Zone Assignment (Relative to Opponent Block)

For each possession phase, players of the attacking team are classified into zones
relative to the **defending team's convex hull block** (Llana et al. 2022 §2.3):

| Zone | Definition |
|------|------------|
| `Inside` | Player is within the convex hull of the defending block |
| `Behind` | Player is beyond the last defensive line (closer to def. goal) |
| `Wing` | Outside the block laterally, but not beyond the last line |

In [ ]:
def point_in_convex_hull(point, hull_points):
    """
    Return True if the 2D point lies inside the convex hull of hull_points.
    Uses a tolerance of 1e-8 to handle boundary cases.
    """
    if len(hull_points) < 3:
        return False
    try:
        hull = ConvexHull(hull_points)
        # A point is inside if all half-space constraints are satisfied
        return bool(np.all(hull.equations @ np.append(point, 1) <= 1e-8))
    except Exception:
        return False


def classify_zone(player_x, defending_positions, last_def_line_x, attack_sign):
    """
    Classify a player's zone relative to the defending block.

    Parameters
    ----------
    player_x           : float  raw x-coordinate of the attacking player
    defending_positions: np.ndarray (n_def, 2)  raw positions of defending outfield players
    last_def_line_x    : float  raw x of the deepest (closest to def. goal) defensive line
    attack_sign        : +1 or −1 for the attacking team

    Returns
    -------
    'Behind' | 'Inside' | 'Wing'
    """
    if np.isnan(player_x) or len(defending_positions) < 3:
        return np.nan

    # 'Behind' = player is farther toward the defending goal than the last defensive line
    # In attack coordinates, the last defensive line is the smallest (most backward)
    # A player "behind" the last line has a larger attack-x than the last_def_line
    # (i.e., they are beyond the last defender toward goal)
    player_atk_x = player_x * attack_sign
    line_atk_x   = last_def_line_x * attack_sign

    if player_atk_x > line_atk_x:
        return 'Behind'

    # Check if inside the convex hull of the defending block
    player_pos = np.array([player_x, 0])  # use x only for 1D hull approx
    if point_in_convex_hull(np.array([player_x, 0]),
                             defending_positions):
        return 'Inside'

    return 'Wing'

In [ ]:
def assign_zones_per_phase(df_phases, df_player_roles,
                            xy_home_full, xy_away_full,
                            half_labels, teamsheets,
                            home_outfield_xids, away_outfield_xids):
    """
    For each possession phase in df_player_roles, assign an attacking zone
    to each attacking player based on the defending team's mean block.
    """
    zone_list = []

    phase_lookup = df_phases.set_index('phase_id')

    for _, prow in df_player_roles.iterrows():
        pid   = prow['phase_id']
        if pid not in phase_lookup.index:
            zone_list.append(np.nan)
            continue

        phase = phase_lookup.loc[pid]
        team  = prow['team']
        half  = phase['half']
        atk_sign = get_attack_sign(half, team)

        s, e = int(phase['start_frame']), int(phase['end_frame']) + 1

        # Mean defending positions
        def_team     = 'Away' if team == 'Home' else 'Home'
        def_xids     = away_outfield_xids if def_team == 'Away' else home_outfield_xids
        xy_def       = xy_away_full if def_team == 'Away' else xy_home_full

        def_mean_pos = []
        for xid in def_xids:
            px = xy_def[s:e, 2 * xid]
            py = xy_def[s:e, 2 * xid + 1]
            if np.mean(~np.isnan(px)) >= 0.5:
                def_mean_pos.append([np.nanmean(px), np.nanmean(py)])

        if len(def_mean_pos) < 3:
            zone_list.append(np.nan)
            continue

        def_pos_arr = np.array(def_mean_pos)

        # Last defensive line = defender with smallest attack-x (deepest in own half)
        def_atk_x = def_pos_arr[:, 0] * atk_sign
        last_def_line_x = def_pos_arr[np.argmin(def_atk_x), 0]

        zone = classify_zone(
            prow['mean_x'], def_pos_arr, last_def_line_x, atk_sign
        )
        zone_list.append(zone)

    df_player_roles = df_player_roles.copy()
    df_player_roles['zone'] = zone_list
    return df_player_roles


df_player_roles = assign_zones_per_phase(
    df_phases, df_player_roles,
    xy_home_full, xy_away_full,
    half_labels, teamsheets,
    home_outfield_xids, away_outfield_xids
)

print(df_player_roles.groupby(['team', 'zone']).size().unstack(fill_value=0))

## Assemble Framewise Context DataFrame

Combine all per-frame and per-phase features into a single `context_df` indexed by frame.

| Column | Description |
|--------|-------------|
| `frame` | Absolute frame number (0 = start of first half) |
| `half` | `'firstHalf'` or `'secondHalf'` |
| `frame_in_half` | Frame number within the half |
| `possession_team` | `'Home'`, `'Away'`, or `'contest'` |
| `ball_live` | Whether the ball is in play |
| `phase_id` | Possession phase index (from `df_phases`) |
| `attack_type` | Phase-level attack type |
| `defense_type` | Frame-level defense classification |
| `home_block_cx/cy` | Home block centroid coordinates |
| `away_block_cx/cy` | Away block centroid coordinates |
| `home_def_line_{1,2,3}` | Home's 3 defensive line x-positions |
| `away_def_line_{1,2,3}` | Away's 3 defensive line x-positions |

In [ ]:
# Build a phase_id per frame using vectorised lookup
frame_phase_id = np.full(n_total, -1, dtype=int)
for _, phase in df_phases.iterrows():
    s = int(phase['start_frame'])
    e = int(phase['end_frame']) + 1
    frame_phase_id[s:e] = int(phase['phase_id'])

# Phase-level lookup series
phase_attack_type = df_phases.set_index('phase_id')['attack_type']

# Frame-in-half index
frame_in_half = np.concatenate([
    np.arange(n_frames_h1),
    np.arange(n_frames_h2)
])

# Vectorised possession decode
poss_decoded = np.array([decode_possession(v) for v in poss_full])
ball_live_arr = (bstat_full == BSTAT_LIVE)

# Attack type per frame (via phase lookup)
atk_type_per_frame = np.array([
    phase_attack_type.get(pid, np.nan) if pid >= 0 else np.nan
    for pid in frame_phase_id
])

context_df = pd.DataFrame({
    'frame':             np.arange(n_total),
    'half':              half_labels,
    'frame_in_half':     frame_in_half,
    'possession_team':   poss_decoded,
    'ball_live':         ball_live_arr,
    'phase_id':          frame_phase_id,
    'attack_type':       atk_type_per_frame,
    'defense_type':      defense_type_arr,
    'home_block_cx':     block_cx_home,
    'home_block_cy':     block_cy_home,
    'away_block_cx':     block_cx_away,
    'away_block_cy':     block_cy_away,
    'home_def_line_1':   def_line_home[:, 0],
    'home_def_line_2':   def_line_home[:, 1],
    'home_def_line_3':   def_line_home[:, 2],
    'away_def_line_1':   def_line_away[:, 0],
    'away_def_line_2':   def_line_away[:, 1],
    'away_def_line_3':   def_line_away[:, 2],
})

print(f"context_df shape: {context_df.shape}")
context_df.head()

In [ ]:
# Summary statistics
print("=== Possession distribution ===")
print(context_df['possession_team'].value_counts())

print("\n=== Attack type distribution (live frames only) ===")
print(context_df.loc[context_df['ball_live'], 'attack_type'].value_counts())

print("\n=== Defense type distribution (live frames only) ===")
print(context_df.loc[context_df['ball_live'], 'defense_type'].value_counts())

## Visualize Block Centroids Over Time

Quick sanity check: block centroids should follow the ball position and swap sides at half time.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 6), sharex=False)

for ax, half, s, e in [
    (axes[0], 'First Half', 0, n_frames_h1),
    (axes[1], 'Second Half', n_frames_h1, n_total)
]:
    frames = np.arange(s, e)
    ax.plot(frames, context_df['home_block_cx'].values[s:e],
            label='Home block centroid x', color='royalblue', alpha=0.7, linewidth=0.8)
    ax.plot(frames, context_df['away_block_cx'].values[s:e],
            label='Away block centroid x', color='tomato', alpha=0.7, linewidth=0.8)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.6)
    ax.set_ylabel('x (m)')
    ax.set_title(half)
    ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## Save Outputs

In [ ]:
os.makedirs(config.CONTEXT_OUTPUT_DIR, exist_ok=True)

context_df.to_csv(f'{config.CONTEXT_OUTPUT_DIR}/{match_id}_context.csv', index=False)
df_phases.to_csv(f'{config.CONTEXT_OUTPUT_DIR}/{match_id}_phases.csv', index=False)
df_player_roles.to_csv(f'{config.CONTEXT_OUTPUT_DIR}/{match_id}_player_roles.csv', index=False)

print(f"Saved context to {config.CONTEXT_OUTPUT_DIR}/{match_id}_context.csv  ({len(context_df)} rows)")
print(f"Saved phases to {config.CONTEXT_OUTPUT_DIR}/{match_id}_phases.csv    ({len(df_phases)} rows)")
print(f"Saved roles  to {config.CONTEXT_OUTPUT_DIR}/{match_id}_player_roles.csv  ({len(df_player_roles)} rows)")